# FHIR Condition Data Quality Profiling

## Purpose

In this notebook, I profile the Silver FHIR Condition dataset and define the
data-quality rules that will later be enforced directly inside the Lakeflow
Bronze-to-Silver transformation.

I am not creating another cleaned Condition table in this notebook.

The purpose of this stage is to:

- identify incomplete or logically invalid Condition records
- distinguish required identifiers from optional FHIR attributes
- validate Condition lifecycle dates
- measure current rule violations
- classify rules as warning, drop, or fail
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.fhir_condition`

### Production design

The final production flow will be:

Bronze  
↓  
FHIR Condition transformation  
+  
Lakeflow expectations  
↓  
Validated Silver Condition  
↓  
Gold

### Data-quality approach

FHIR Condition records can legitimately omit some optional attributes such as
an Encounter reference or an abatement date.

I therefore treat missing optional fields as monitoring concerns rather than
automatic rejection conditions.

The most important fields for relational integrity are the Condition ID,
Patient ID, and Condition code.

In [0]:
# loading the current Silver Condition dataset for quality profiling.

from pyspark.sql import functions as F

CONDITION_TABLE = "health_insurance.silver.fhir_condition"

condition_df = spark.table(CONDITION_TABLE)

print(f"Condition rows: {condition_df.count():,}")

condition_df.printSchema()

display(condition_df.limit(10))

In [0]:
# defining candidate Condition quality rules by severity.

CONDITION_WARN_RULES = {
    "encounter_reference_present":
        "encounter_id IS NOT NULL",

    "condition_name_present":
        "condition_name IS NOT NULL",

    "clinical_status_present":
        "clinical_status IS NOT NULL",

    "verification_status_present":
        "verification_status IS NOT NULL",

    "onset_datetime_present":
        "onset_datetime IS NOT NULL"
}


CONDITION_DROP_RULES = {
    "condition_id_present":
        "condition_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "condition_code_present":
        "condition_code IS NOT NULL"
}


CONDITION_FAIL_RULES = {
    "condition_timeline_valid":
        """
        onset_datetime IS NULL
        OR abatement_datetime IS NULL
        OR abatement_datetime >= onset_datetime
        """
}

In [0]:
# measuring how many Condition rows violate each candidate quality rule.

def profile_rules(df, rules, severity):

    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(
                f"NOT ({condition}) OR ({condition}) IS NULL"
            )
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition.strip(),
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
# profiling all proposed Condition quality rules.

condition_quality_results = []

condition_quality_results += profile_rules(
    condition_df,
    CONDITION_WARN_RULES,
    "WARN"
)

condition_quality_results += profile_rules(
    condition_df,
    CONDITION_DROP_RULES,
    "DROP"
)

condition_quality_results += profile_rules(
    condition_df,
    CONDITION_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the Condition quality profile as a structured result.

condition_quality_profile_df = spark.createDataFrame(
    condition_quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    condition_quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# inspecting Condition timeline anomalies directly.

condition_df.select(
    F.count("*").alias("total_conditions"),

    F.sum(
        F.col("onset_datetime").isNull().cast("int")
    ).alias("missing_onset"),

    F.sum(
        F.col("abatement_datetime").isNull().cast("int")
    ).alias("missing_abatement"),

    F.sum(
        (
            F.col("abatement_datetime")
            < F.col("onset_datetime")
        ).cast("int")
    ).alias("abatement_before_onset")
).show()

In [0]:
# inspecting the actual Condition status domains
# before finalizing controlled-value quality rules.

display(
    condition_df
    .groupBy(
        "clinical_status",
        "verification_status"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

In [0]:
# defining the finalized Condition quality contract.

CONDITION_WARN_RULES = {
    "encounter_reference_present":
        "encounter_id IS NOT NULL",

    "condition_name_present":
        "condition_name IS NOT NULL",

    "recognized_clinical_status":
        "clinical_status IN ('ACTIVE', 'RESOLVED')",

    "recognized_verification_status":
        "verification_status IN ('CONFIRMED')",

    "onset_datetime_present":
        "onset_datetime IS NOT NULL"
}


CONDITION_DROP_RULES = {
    "condition_id_present":
        "condition_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "condition_code_present":
        "condition_code IS NOT NULL"
}


CONDITION_FAIL_RULES = {
    "condition_timeline_valid":
        """
        onset_datetime IS NULL
        OR abatement_datetime IS NULL
        OR abatement_datetime >= onset_datetime
        """
}

In [0]:
# preparing my finalized Condition quality rules
# for storage in the reusable quality-rules module.

condition_quality_code = '''
# === CONDITION QUALITY RULES START ===

CONDITION_WARN_RULES = {
    "encounter_reference_present":
        "encounter_id IS NOT NULL",

    "condition_name_present":
        "condition_name IS NOT NULL",

    "recognized_clinical_status":
        "clinical_status IN ('ACTIVE', 'RESOLVED')",

    "recognized_verification_status":
        "verification_status IN ('CONFIRMED')",

    "onset_datetime_present":
        "onset_datetime IS NOT NULL"
}


CONDITION_DROP_RULES = {
    "condition_id_present":
        "condition_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "condition_code_present":
        "condition_code IS NOT NULL"
}


CONDITION_FAIL_RULES = {
    "condition_timeline_valid":
        """
        onset_datetime IS NULL
        OR abatement_datetime IS NULL
        OR abatement_datetime >= onset_datetime
        """
}

# === CONDITION QUALITY RULES END ===
'''

In [0]:
# adding or updating the Condition section
# without changing my existing Claims or Patient rules.

from pathlib import Path
import re

QUALITY_RULES_PATH = (
    "/Workspace/Khaoula healthy insurance project/"
    "Khaoula-healthy-insuarance-project/"
    "04-data-quality/"
    "quality_rules.py"
)

quality_file = Path(QUALITY_RULES_PATH)

existing_code = quality_file.read_text(
    encoding="utf-8"
)

start_marker = "# === CONDITION QUALITY RULES START ==="
end_marker = "# === CONDITION QUALITY RULES END ==="

pattern = (
    re.escape(start_marker)
    + r".*?"
    + re.escape(end_marker)
)

if start_marker in existing_code:
    updated_code = re.sub(
        pattern,
        condition_quality_code.strip(),
        existing_code,
        flags=re.DOTALL
    )
else:
    updated_code = (
        existing_code.rstrip()
        + "\n\n\n"
        + condition_quality_code.strip()
        + "\n"
    )

quality_file.write_text(
    updated_code,
    encoding="utf-8"
)

print("Condition quality rules saved successfully.")